# Task-by-Task Testing

Test each competition task type against the local agent using prompts observed from competition runs.

**Usage**: Start the server with `uv run uvicorn src.main:app --port 8000`, then run cells below.

In [20]:
import httpx
import os
import json
from dotenv import load_dotenv

load_dotenv("../.env")

AGENT_URL = "https://sam-reduction-acceptable-cap.trycloudflare.com"
SANDBOX_BASE_URL = os.getenv("API_URL", "https://kkpqfuj-amager.tripletex.dev/v2")
SESSION_TOKEN = os.getenv("SESSION_TOKEN", "")
API_KEY = os.getenv("AGENT_API_KEY", "")
print(f"Using API key: xxx..." if API_KEY else "No API key provided, proceeding without authentication.")

HEADER = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

def send_solve(prompt: str, files: list = None, base_url: str = None, token: str = None) -> dict:
    """Send a task to the local /solve endpoint."""
    payload = {
        "prompt": prompt,
        "files": files or [],
        "tripletex_credentials": {
            "base_url": base_url or SANDBOX_BASE_URL,
            "session_token": token or SESSION_TOKEN,
        },
    }
    with httpx.Client(timeout=300) as client:
        resp = client.post(f"{AGENT_URL}/solve", json=payload, headers=HEADER)
        print(f"Status: {resp.status_code} | Response: {resp.json()}")
        return resp.json()


def query(path: str, params: dict = None) -> dict:
    """Query sandbox Tripletex API directly."""
    with httpx.Client(timeout=30) as client:
        resp = client.get(
            f"{SANDBOX_BASE_URL}{path}",
            auth=("0", SESSION_TOKEN),
            params=params or {"fields": "*", "count": 100},
            headers=HEADER
        )
        return resp.json()


print(f"Sandbox: {SANDBOX_BASE_URL}")
print(f"Token: {SESSION_TOKEN[:20]}..." if SESSION_TOKEN else "No token!")

Using API key: xxx...
Sandbox: https://kkpqfuj-amager.tripletex.dev/v2
Token: eyJ0b2tlbklkIjoyMTQ3...


## Task 1 — Create Departments (Score: 1.08, Tier 1)
Optimal: 3 POST calls, 0 errors

In [22]:
send_solve("Créez trois départements dans Tripletex : \"Utvikling\", \"Kundeservice\" et \"Innkjøp\".")

Status: 200 | Response: {'status': 'completed'}


{'status': 'completed'}

In [8]:
# Verify
print(json.dumps(query("/department", {"fields": "id,name,departmentNumber"}), indent=2, ensure_ascii=False))

{
  "fullResultSize": 1,
  "from": 0,
  "count": 1,
  "versionDigest": "'If-None-Match' header not specified",
  "values": [
    {
      "id": 841310,
      "name": "Avdeling",
      "departmentNumber": ""
    }
  ]
}


## Task 2 — Create Customer (Score: 2.00 PERFECT, Tier 1)
Optimal: 1 POST call, 0 errors

In [ ]:
send_solve("Crie o cliente Floresta Lda com número de organização 893475656. O endereço é Kirkegata 132, 7010 Trondheim. E-mail: post@floresta.no.")

## Task 3 — Create Supplier (Score: 1.50, Tier 1)
FIX: Set both `email` AND `invoiceEmail`. Optimal: 1 POST call, 0 errors

In [ ]:
send_solve("Enregistrez le fournisseur Cascade SARL avec le numéro d'organisation 997712560. E-mail : faktura@cascadesarl.no.")

In [ ]:
# Verify - check both email and invoiceEmail are set
print(json.dumps(query("/customer", {"fields": "id,name,email,invoiceEmail,isSupplier,isCustomer,organizationNumber", "isSupplier": True}), indent=2, ensure_ascii=False))

## Task 4 — Employee with Employment (Score: 0.86, Tier 1)
FIX: userType=STANDARD, department required, correct employment schema. Optimal: 3 calls

In [ ]:
send_solve("Me har ein ny tilsett som heiter Geir Neset, fødd 24. June 1997. Opprett vedkomande som tilsett med e-post geir.neset@example.org og startdato 15. October 2026.")

In [ ]:
# Verify employee + employment
employees = query("/employee", {"fields": "id,firstName,lastName,email,dateOfBirth"})
print("Employees:", json.dumps(employees, indent=2, ensure_ascii=False))
# Check employment records
for emp in employees.get("values", []):
    if emp.get("firstName") == "Geir":
        employment = query(f"/employee/employment", {"employeeId": emp["id"], "fields": "*"})
        print(f"\nEmployment for {emp['firstName']}:", json.dumps(employment, indent=2, ensure_ascii=False))

## Task 5 — Create Departments (Score: 1.33, Tier 2?)
This was originally labeled Task 5 but seems to be same as Task 1. May need more data.

## Task 6 — Create and Send Invoice (Score: 6/7 checks, Tier 2)
FIX: Use vatType id=3 (25%) not id=6 (0%). "sem IVA" = price stated excl VAT, not 0% VAT.
Also need bank account setup for company.

In [ ]:
send_solve("Crie e envie uma fatura ao cliente Rio Azul Lda (org. nº 959230277) por 27900 NOK sem IVA. A fatura refere-se a Relatório de análise.")

In [ ]:
# Verify invoice - check amount includes VAT
invoices = query("/invoice", {"invoiceDateFrom": "2020-01-01", "invoiceDateTo": "2099-12-31", "fields": "id,invoiceNumber,amount,amountExcludingVat,amountOutstanding,customer(name)"})
print(json.dumps(invoices, indent=2, ensure_ascii=False))

## Task 7 — Invoice with Payment (Score: 2/7, Tier 2)
CRITICAL FIX: "has an outstanding invoice" = EXISTING invoice. Search for it, don't create new.
Also: deliveryDate required on orders.

In [ ]:
# NOTE: This task requires pre-populated data (customer + invoice already exist).
# On the sandbox this won't work as-is. In competition, the validator pre-creates them.
send_solve("The customer Ironbridge Ltd (org no. 948020890) has an outstanding invoice for 6300 NOK excluding VAT for \"Consulting Hours\". Register full payment on this invoice.")

## Task 8 — Create Project (Score: 0/7, Tier 2)
FIX: Ensure employee email is set. startDate required. Search employee first.

In [ ]:
send_solve("Erstellen Sie das Projekt \"Analyse Windkraft\" verknüpft mit dem Kunden Windkraft GmbH (Org.-Nr. 897356171). Projektleiter ist Finn Richter (finn.richter@example.org).")

In [ ]:
# Verify project, customer, and employee email
projects = query("/project", {"fields": "id,name,customer(name,organizationNumber),projectManager(firstName,lastName,email),startDate"})
print("Projects:", json.dumps(projects, indent=2, ensure_ascii=False))

# Also check employee has email set
employees = query("/employee", {"fields": "id,firstName,lastName,email"})
print("\nEmployees:", json.dumps(employees, indent=2, ensure_ascii=False))